# 4. Classificação pelos eixos da BNCC
A classificação é baseada na presença de termos e expressões no texto normalizado. Um artigo pode receber mais de um eixo. Todas as evidências serão registradas para revisão.

In [1]:
from pathlib import Path
import sys
import pandas as pd

inicio = Path.cwd().resolve()
raiz = next((p for p in (inicio, *inicio.parents) if (p / 'README.md').exists() and (p / 'dados').exists()), None)
if raiz is None:
    raise FileNotFoundError('Não foi possível localizar a pasta do projeto.')
sys.path.insert(0, str(raiz))

from apoio.funcoes import carregar_eixos_bncc, selecionar_descritores_relevantes, encontrar_evidencias_bncc
pasta_processados = raiz / 'dados' / '1_processados'

## 4.1 Leitura dos artigos e do vocabulário

In [2]:
# Carregar os dados processados
df = pd.read_csv(pasta_processados / '02_artigos_pre_processados.csv', encoding='utf-8-sig')
ranking_termos = pd.read_csv(pasta_processados / '03_ranking_termos_titulos.csv', encoding='utf-8-sig')
ranking_bigramas = pd.read_csv(pasta_processados / '03_ranking_bigramas_titulos.csv', encoding='utf-8-sig')
eixos_candidatos = carregar_eixos_bncc(raiz / 'apoio' / 'termos_bncc.yml')

if 'evento' not in df.columns:
    raise ValueError("A coluna 'evento' é obrigatória para a classificação segmentada por evento.")

# Seleciona descritores relevantes para cada evento, em vez de usar um vocabulário global

def selecionar_eixos_por_evento(evento: str) -> dict[str, dict[str, object]]:
    termos_evento = set(ranking_termos.loc[ranking_termos['evento'] == evento, 'termo'])
    bigramas_evento = set(ranking_bigramas.loc[ranking_bigramas['evento'] == evento, 'bigrama'])
    return selecionar_descritores_relevantes(eixos_candidatos, termos_evento, bigramas_evento)

# Mantém o vocabulário de cada evento para a classificação
vocabularios_por_evento = {
    evento: selecionar_eixos_por_evento(evento)
    for evento in sorted(df['evento'].dropna().astype(str).str.strip().unique())
}

print(f'Artigos recebidos: {len(df)}')
print('Eventos disponíveis:', sorted(vocabularios_por_evento.keys()))

# Criar DataFrame com os resultados por evento
pd.DataFrame([
    {'evento': evento, 'codigo': codigo, 'eixo': dados['nome'], 'termos_ativos': len(dados['termos']), 'bigramas_ativos': len(dados['bigramas'])}
    for evento, eixos in vocabularios_por_evento.items()
    for codigo, dados in eixos.items()
])

Artigos recebidos: 2108
Eventos disponíveis: ['WEI', 'WIE']


,evento,codigo,eixo,termos_ativos,bigramas_ativos
0,WEI,pensamento_computacional,Pensamento Computacional,8,6
1,WEI,mundo_digital,Mundo Digital,5,3
2,WEI,cultura_digital,Cultura Digital,2,0
3,WIE,pensamento_computacional,Pensamento Computacional,7,5
4,WIE,mundo_digital,Mundo Digital,4,1
5,WIE,cultura_digital,Cultura Digital,4,4


## 4.2 Aplicação das regras
A busca usa palavras e expressões completas. Por exemplo, o termo `tic` não será encontrado dentro de outra palavra.

In [ ]:
# Classifica cada artigo usando o vocabulário específico do seu evento

def classificar_artigo_por_evento(linha: pd.Series) -> dict[str, dict[str, list[str]]]:
    evento = str(linha['evento']).strip()
    eixos_evento = vocabularios_por_evento.get(evento, {})
    return encontrar_evidencias_bncc(str(linha['texto_limpo']).strip(), eixos_evento)


def listar_nomes_eixos(resultado: dict[str, dict[str, list[str]]], evento: str) -> str:
    eixos_evento = vocabularios_por_evento.get(evento, {})
    return '; '.join(eixos_evento[codigo]['nome'] for codigo in resultado)


df['evidencias_bncc'] = df.apply(classificar_artigo_por_evento, axis=1)
df['eixos_bncc'] = df.apply(
    lambda linha: listar_nomes_eixos(linha['evidencias_bncc'], str(linha['evento']).strip()),
    axis=1,
)
df['quantidade_eixos'] = df['evidencias_bncc'].apply(len)
df[['id_artigo', 'evento', 'titulo', 'eixos_bncc', 'quantidade_eixos']].head(10)

NameError: name 'eixos' is not defined

## 4.3 Tabela de classificações e evidências

In [ ]:
linhas_classificacao = []
# Gerar linhas de classificação para cada artigo e cada eixo
for _, artigo in df.iterrows():
    for codigo_eixo, evidencias in artigo['evidencias_bncc'].items():
        linhas_classificacao.append({
            'id_artigo': artigo['id_artigo'],
            'evento': artigo['evento'],
            'ano': artigo['ano'],
            'eixo_bncc': vocabularios_por_evento[str(artigo['evento']).strip()][codigo_eixo]['nome'],
            'quantidade_evidencias': len(evidencias['termos']) + len(evidencias['bigramas']),
            'termos_encontrados': ' | '.join(evidencias['termos']),
            'bigramas_encontrados': ' | '.join(evidencias['bigramas']),
        })

# Criar DataFrame com as classificações
classificacoes = pd.DataFrame(linhas_classificacao)
classificacoes.head(10)

# Gerar resumo por eixo
resumo_eixos = (classificacoes.groupby(['evento', 'eixo_bncc'])['id_artigo']
    .nunique()
    .rename('quantidade_artigos')
    .reset_index())
resumo_eixos['percentual_corpus'] = (resumo_eixos['quantidade_artigos'] / len(df) * 100).round(2)
resumo_eixos.sort_values(['evento', 'quantidade_artigos'], ascending=[True, False])

In [ ]:
# Gerar resumo por eixo
resumo_eixos = (classificacoes.groupby('eixo_bncc')['id_artigo']
    .nunique()
    .rename('quantidade_artigos')
    .reset_index())
resumo_eixos['percentual_corpus'] = (resumo_eixos['quantidade_artigos'] / len(df) * 100).round(2)
resumo_eixos.sort_values('quantidade_artigos', ascending=False)

In [ ]:
# Gerar resumo por quantidade de eixos
resumo_sobreposicao = (df['quantidade_eixos']
    .value_counts()
    .sort_index()
    .rename_axis('quantidade_eixos')
    .reset_index(name='quantidade_artigos'))
resumo_sobreposicao['percentual_corpus'] = (resumo_sobreposicao['quantidade_artigos'] / len(df) * 100).round(2)
resumo_sobreposicao

## 4.5 Artigos não classificados

In [ ]:
nao_classificados = df[df['quantidade_eixos'] == 0].copy()
print(f'Não classificados: {len(nao_classificados)} de {len(df)}')
nao_classificados[['id_artigo', 'evento', 'ano', 'titulo']].head(20)

## 4.6 Descritores que mais produziram classificações
Esta tabela ajuda a identificar termos excessivamente amplos ou pouco utilizados.

In [ ]:
evidencias_termos = (classificacoes[['id_artigo', 'evento', 'eixo_bncc', 'termos_encontrados']]
    .assign(tipo='termo', descritor=lambda x: x['termos_encontrados'].fillna('').str.split(' | ', regex=False))
    .explode('descritor').query("descritor != ''"))
evidencias_bigramas = (classificacoes[['id_artigo', 'evento', 'eixo_bncc', 'bigramas_encontrados']]
    .assign(tipo='bigrama', descritor=lambda x: x['bigramas_encontrados'].fillna('').str.split(' | ', regex=False))
    .explode('descritor').query("descritor != ''"))
evidencias_descritores = pd.concat([
    evidencias_termos[['id_artigo', 'evento', 'eixo_bncc', 'tipo', 'descritor']],
    evidencias_bigramas[['id_artigo', 'evento', 'eixo_bncc', 'tipo', 'descritor']],
])
frequencia_descritores = (evidencias_descritores.groupby(['evento', 'eixo_bncc', 'tipo', 'descritor'])['id_artigo']
    .nunique()
    .rename('quantidade_artigos')
    .reset_index()
    .sort_values(['evento', 'eixo_bncc', 'quantidade_artigos'], ascending=[True, True, False]))
frequencia_descritores.groupby(['evento', 'eixo_bncc']).head(10)

## 4.7 Exportação para validação

In [ ]:
artigos_classificados = df.drop(columns='evidencias_bncc')
artigos_classificados.to_csv(pasta_processados / '04_artigos_classificados_bncc.csv', index=False, encoding='utf-8-sig')
classificacoes.to_csv(pasta_processados / '04_classificacoes_bncc.csv', index=False, encoding='utf-8-sig')
nao_classificados.drop(columns='evidencias_bncc').to_csv(pasta_processados / '04_artigos_nao_classificados.csv', index=False, encoding='utf-8-sig')
resumo_eixos.to_csv(pasta_processados / '04_resumo_classificacao_bncc.csv', index=False, encoding='utf-8-sig')
frequencia_descritores.to_csv(pasta_processados / '04_frequencia_descritores_bncc.csv', index=False, encoding='utf-8-sig')
print('Arquivos de validação exportados.')